# Calculate the data statistics from image downloads

In [ ]:
from types import SimpleNamespace
import argparse
import sys
import os

from calculate_data_stats import check_stats, setup_logger

In [ ]:
if sys.platform.startswith("linux"):
    data_dir = "/gws/ssde/j25b/ceh_generic/kgoldmann/"
    project_dir = "/home/users/katriona/gbif_download_standalone"
elif sys.platform == "darwin":
    data_dir = "/Users/lbokeria/Documents/projects/gbif-species-trainer-data/"
    project_dir = "/Users/lbokeria/Documents/projects/gbif_download_standalone/"
else:
    print("Not linux or mac!")



In [ ]:
gbif_img_dir     =os.path.join(data_dir,"gbif_images")
print(gbif_img_dir)

In [ ]:
checklists = os.listdir(os.path.join(project_dir, "species_checklists/final"))
print(checklists)

In [ ]:
for checklist_name in checklists:
    if checklist_name.endswith("-keys-nodup.csv"):
        print(f"Processing {checklist_name}")
        args = SimpleNamespace(
            write_directory  =os.path.join(project_dir,"data_stats_files"),
            species_checklist=os.path.join(project_dir,"species_checklists", "final",
                                           checklist_name),
            gbif_img_dir     =os.path.join(data_dir,"gbif_images")

        )
        check_stats(args)

# Deal with the error cases

In [ ]:
# for each /gws/ssde/j25b/ceh_generic/kgoldmann/gbif_images/*/*/*/ directory which doesnt have meta_data.json file, remove the directory
for root, dirs, files in os.walk(gbif_img_dir):
    for dir_name in dirs:
        dir_path = os.path.join(root, dir_name)
        meta_file = os.path.join(dir_path, "meta_data.json")
        if not os.path.exists(meta_file):
            print(f"Removing directory {dir_path} as meta_data.json does not exist")
            # os.rmdir(dir_path)

In [ ]:
import shutil

In [ ]:
# for each /gws/ssde/j25b/ceh_generic/kgoldmann/gbif_images/*/*/*/meta_data.json
# where the number of entries in meta_data.json does not equal the number of
# images in the directory, remove the directory
import json

problem_species = 0
for root, dirs, files in os.walk(gbif_img_dir):
    for dir_name in dirs:
        dir_path = os.path.join(root, dir_name)
        meta_file = os.path.join(dir_path, "meta_data.json")
        if os.path.exists(meta_file):
            with open(meta_file, "r") as f:
                lines = f.readlines()
            try:
                meta_dict = json.loads(lines[0])
            except json.JSONDecodeError:
                continue
            num_meta = len(meta_dict)
            num_images = len([f for f in os.listdir(dir_path) if f.endswith(('.jpg', '.jpeg', '.png'))])
            if num_meta != num_images:
                # print(f"Removing directory {os.path.dirname(dir_path)} as meta_data.json entries ({num_meta}) do not equal number of images ({num_images})")
                problem_species += 1
                shutil.rmtree(dir_path, ignore_errors=True)

print(f"Total problem species directories removed: {problem_species}")

In [ ]:
find . ! -name . -prune -type d -exec sh -c '
  for dir do
    dir="${dir#./}"
    printf "%s:\t" "$dir"
    find ".//$dir" -type f | grep -c //
  done' sh {} +


# Take a look at the file

In [ ]:
project_dir

In [ ]:
import pandas as pd
checklist_name = "costarica-moths-keys-nodup"
data_stats = pd.read_csv(os.path.join(project_dir,"data_stats_files","data_stats_"+checklist_name+".csv"))

data_stats.head()